## 1. Load and Inspect Silver NO2 Data

Load hourly NO2 measurements from the Silver layer and inspect the schema, record count, timestamps, and pollutant values before daily aggregation.

In [ ]:
df_no2 = spark.table("silver_no2_hourly")

df_no2.printSchema()
print("Total rows:", df_no2.count())

In [ ]:
df_no2.select(
    "measurement_time_utc",
    "measurement_time_local",
    "no2_value"
).show(5, truncate=False)

## 2. Parse Measurement Date

Extract the calendar date from local measurement timestamps and convert it to DateType.

Validate the conversion by checking for NULL dates.

In [ ]:
from pyspark.sql import functions as F

df_no2 = df_no2.withColumn(
    "measurement_date",
    F.to_date(F.split(F.col("measurement_time_local"), " ").getItem(0), "M/d/yyyy")
)

df_no2.select(
    "measurement_time_local",
    "measurement_date"
).show(5, truncate=False)

df_no2.filter(
    F.col("measurement_date").isNull()
).count()


## 3. Aggregate Daily NO2 Measurements

Aggregate hourly NO2 measurements by date to calculate daily average concentration, maximum concentration, and measurement count.

Inspect the date range before applying the reporting period filter.

In [ ]:
df_no2_daily = (
    df_no2.groupBy("measurement_date")
    .agg(
        F.round(F.avg("no2_value"), 4).alias("avg_no2_ppm"),
        F.round(F.max("no2_value"), 4).alias("max_no2_ppm"),
        F.count("no2_value").alias("measurement_count")
    )
    .orderBy("measurement_date")
)

df_no2_daily.show(10, truncate=False)
print("Total days:", df_no2_daily.count())

In [ ]:
df_no2_daily.orderBy(
    F.col("measurement_date").desc()
).show(5, truncate=False)

## 4. Filter Reporting Period

Filter daily NO2 measurements to June and July 2025 to align with the NYC Taxi analysis period.

Exclude records outside the reporting window and validate the final date range.

In [ ]:
df_no2_daily = df_no2_daily.filter(
    (F.col("measurement_date") >= F.lit("2025-06-01")) &
    (F.col("measurement_date") < F.lit("2025-08-01"))
)

print("Total days:", df_no2_daily.count())

df_no2_daily.orderBy(
    F.col("measurement_date").desc()
).show(3, truncate=False)

## 5. Validate Duplicate NO2 Measurements

Check for duplicate measurements using the combination of sensor_id and measurement_time_utc to ensure source data consistency.

In [ ]:
df_no2.groupBy(
    "sensor_id",
    "measurement_time_utc"
).count().filter(
    F.col("count") > 1
).show(10, truncate=False)

## 6. Save and Verify Gold NO2 Daily

Save the validated daily NO2 aggregates as a Delta table in the Gold layer.

Verify the saved row count, schema, and sample records.

In [ ]:
df_no2_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_no2_daily")

In [ ]:
df_gold = spark.table("gold_no2_daily")

print("Total days:", df_gold.count())
df_gold.printSchema()
df_gold.show(5)